<a href="https://colab.research.google.com/github/pelamiguelpr/pelamiguelpr-DataSience-GenAI-Submissions/blob/main/Assignment_8/6_02_DNN_101_COMPLETED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [ ]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [ ]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [ ]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [ ]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [ ]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [ ]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 27575.4863
Epoch [10/100], Loss: 35397.7383
Epoch [10/100], Loss: 33949.9219
Epoch [10/100], Loss: 30950.7637
Epoch [10/100], Loss: 26751.2695
Epoch [10/100], Loss: 27165.0703
Epoch [10/100], Loss: 25908.9336
Epoch [10/100], Loss: 24869.4902
Epoch [20/100], Loss: 27879.1074
Epoch [20/100], Loss: 25563.207
Epoch [20/100], Loss: 36879.5156
Epoch [20/100], Loss: 28683.6621
Epoch [20/100], Loss: 29036.8242
Epoch [20/100], Loss: 24846.1543
Epoch [20/100], Loss: 32634.0273
Epoch [20/100], Loss: 56591.5625
Epoch [30/100], Loss: 29928.2051
Epoch [30/100], Loss: 29282.3496
Epoch [30/100], Loss: 23348.0117
Epoch [30/100], Loss: 33627.3281
Epoch [30/100], Loss: 33115.3828
Epoch [30/100], Loss: 26770.4453
Epoch [30/100], Loss: 29144.4023
Epoch [30/100], Loss: 57681.2812
Epoch [40/100], Loss: 25846.0918
Epoch [40/100], Loss: 28116.7871
Epoch [40/100], Loss: 36049.1523
Epoch [40/100], Loss: 26323.5859
Epoch [40/100], Loss: 28290.125
Epoch [40/100], Loss: 29234.0645
Epoch [40/10

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [ ]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 25877.7841796875


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [ ]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,1.060439,219.0
1,1.060439,70.0
2,1.060439,202.0
3,1.060439,230.0
4,1.060439,111.0
...,...,...
84,1.060439,153.0
85,1.060439,98.0
86,1.060439,37.0
87,1.060439,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

In [ ]:
def train_model(model, epochs=1000, lr=0.001, verbose=True):
    model.to(device)
    optimiser = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimiser.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimiser.step()
            running_loss += loss.item() * len(inputs)
        if verbose and (epoch+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Avg loss: {round(running_loss / len(train_dataset))}')
    return model

def test_mse(model):
    model.eval()
    preds, acts = [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            preds.append(model(inputs.to(device)).cpu())
            acts.append(targets)
    return nn.MSELoss()(torch.cat(preds), torch.cat(acts)).item()

torch.manual_seed(42)
model = train_model(DiabetesModel(), epochs=1000)
print(f'Test MSE: {round(test_mse(model))}')

Epoch [100/1000], Avg loss: 26770
Epoch [200/1000], Avg loss: 15494
Epoch [300/1000], Avg loss: 5905
Epoch [400/1000], Avg loss: 4130
Epoch [500/1000], Avg loss: 3878
Epoch [600/1000], Avg loss: 3690
Epoch [700/1000], Avg loss: 3532
Epoch [800/1000], Avg loss: 3409
Epoch [900/1000], Avg loss: 3303
Epoch [1000/1000], Avg loss: 3215
Test MSE: 2994


Increasing the number of epochs from 100 to 1,000 improved the model dramatically, reducing the test MSE from 23,308 to 2,994. With only 100 epochs, the model had not learned any meaningful pattern and was effectively predicting the same value for every patient. With 1,000 epochs, the average training loss fell steadily, from 26,770 at epoch 100 to 3,215 at epoch 1,000, and the model now performs much better than simply predicting the average for every patient (MSE of 5,362).
However, two caveats apply. First, the loss was still decreasing slightly at epoch 1,000, which suggests the model had not fully converged and could improve further with more training or a higher learning rate. Second, the test MSE of 2,994 is close to that of a simple linear regression (2,900), so the neural network has not yet demonstrated any advantage over a much simpler model. The training loss (3,215) and test MSE (2,994) are similar, which indicates the model is not overfitting.


In [ ]:
class FlexibleModel(nn.Module):
    def __init__(self, hidden_sizes):
        super().__init__()
        layers, in_size = [], 10
        for h in hidden_sizes:
            layers += [nn.Linear(in_size, h), nn.ReLU()]
            in_size = h
        layers.append(nn.Linear(in_size, 1))
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

for sizes in [[5, 5], [16, 8], [32, 16], [64, 32, 16]]:
    torch.manual_seed(42)
    m = train_model(FlexibleModel(sizes), epochs=1000, lr=0.01, verbose=False)
    print(f'{sizes}: test MSE = {round(test_mse(m))}')

[5, 5]: test MSE = 2867
[16, 8]: test MSE = 2923
[32, 16]: test MSE = 2603
[64, 32, 16]: test MSE = 2844


Varying the architecture produced only modest differences, with test MSEs ranging from 2,603 to 2,923. The best result came from two hidden layers of 32 and 16 neurons (MSE of 2,603), around 10% better than linear regression. Adding a third layer ([64, 32, 16]) did not improve performance further, suggesting that additional complexity offers little benefit for a dataset of only 353 training observations and 10 features.
It is not possible to identify a truly optimal architecture from these results. The differences between models are small relative to the variability expected from a test set of only 89 patients, and each model was trained only once with a single random seed, so a different seed could change the ranking. Moreover, selecting the architecture based on test performance means the test set is no longer an unbiased measure of performance; a more rigorous approach would compare architectures using a separate validation set or cross-validation. Overall, the results suggest that for small, structured datasets such as this, a neural network offers at best a limited improvement over simpler models such as linear regression.
